In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
# Load an instruction-tuned model via Hugging Face Transformers
MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"

In [4]:
print(f"Loading tokenizer and model: {MODEL_ID}...")

Loading tokenizer and model: Qwen/Qwen2.5-Coder-3B-Instruct...


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
print("Model loaded succesfully")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded succesfully


In [7]:
def run_llm(messages: list[dict], max_new_tokens: int = 512) -> str:
  """Helper utility to format chat messages and generate an inference response."""
  prompt = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
  )
  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

  with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.2,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
  generated_tokens = outputs[0][inputs.input_ids.shape[1]:]
  return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

# ==============================================================================
# Define Generator and Reflector System Prompts
# ==============================================================================

In [9]:
GENERATOR_SYSTEM_PROMPT = """You are an expert Python developer.
Your goal is to write clean, correct, and efficient Python solutions based on the request and any feedback provided."""

In [10]:
REFLECTOR_SYSTEM_PROMPT = """You are an elite Code Reviewer and QA Engineer.
Critically evaluate the candidate Python code. Check for:
1. Logic errors or off-by-one errors.
2. Edge case failures (e.g., empty lists, negative inputs, null values).
3. Type safety, efficiency, and clarity.

If the solution is fully optimal and bug-free, respond with EXACTLY the word 'APPROVED'.
Otherwise, list specific, actionable criticisms for improvement."""

# ==============================================================================
# The Reflective Agent Loop Implementation
# ==============================================================================

In [11]:
def reflective_code_agent(task_description: str, max_iterations: int = 3) -> str:
  """Executes the Reflection Pattern: Generate -> Critique -> Refine."""
  print(f"\n--- Starting Reflection Agent for Task ---")
  print(f"Goal: {task_description}\n")
  current_code = ""
  feedback = ""

  for i in range(1, max_iterations + 1):
    print(f"== Iteration {i}/{max_iterations} ==")

    # 1. GENERATION STEP
    gen_messages = [{"role": "system", "content": GENERATOR_SYSTEM_PROMPT}]
    if i == 1:
      gen_messages.append({"role": "user", "content": f"Task: {task_description}"})
    else:
      gen_prompt = (
          f"Task: {task_description}\n\n"
          f"Previous Implementation:\n```python\n{current_code}\n```\n\n"
          f"Critique and Feedback received:\n{feedback}\n\n"
          f"Please update and output the revised Python solution addressing all criticisms."
      )
      gen_messages.append({"role": "user", "content": gen_prompt})
    print("[Generator] Drafting solution...")
    current_code = run_llm(gen_messages)
    print(f"\n[Generator Output]:\n{current_code}\n")

    # 2.Reflection step
    ref_messages = [
        {"role": "system", "content": REFLECTOR_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Task: {task_description}\n\nCandidate Code to Review:\n{current_code}"

        }
    ]
    print("[Reflector] Evaluating candidate solution...")
    critique = run_llm(ref_messages)
    print(f"\n[Reflector Feedback]:\n{critique}\n")

    # 3.Verification & Early Stopping
    if "APPROVED" in critique.upper():
      print(f"✓ Solution approved by Reflector on iteration {i}!")
      break
    else:
      feedback = critique

  return current_code

# ==============================================================================
# Execute the Reflection Loop
# ==============================================================================

In [12]:
user_task = "Write a Python function to find the length of the longest substring without repeating characters."
final_solution = reflective_code_agent(task_description=user_task, max_iterations=3)
print("================ FINAL OUTPUT ================")
print(final_solution)


--- Starting Reflection Agent for Task ---
Goal: Write a Python function to find the length of the longest substring without repeating characters.

== Iteration 1/3 ==
[Generator] Drafting solution...

[Generator Output]:
To solve the problem of finding the length of the longest substring without repeating characters in Python, we can use a sliding window approach. This method efficiently tracks the current substring and updates the maximum length found as we iterate through the string.

Here's a step-by-step explanation of the approach:

1. **Initialize Variables**: We need a dictionary to keep track of the last seen index of each character. Also, we'll use two pointers, `start` and `end`, to represent the current window of characters being considered.

2. **Iterate Through the String**: Use the `end` pointer to expand the window by adding characters to the dictionary.

3. **Check for Repeating Characters**: If a character is already in the dictionary and its last seen index is withi